## 介紹 

本課程將涵蓋： 
- 什麼是函數調用及其使用場景 
- 如何使用 OpenAI 創建函數調用 
- 如何將函數調用整合到應用程式中 

## 學習目標 

完成本課程後，你將知道如何並理解： 

- 使用函數調用的目的 
- 使用 OpenAI 服務設定函數調用 
- 設計有效的函數調用以符合應用程式的使用需求 


## 理解函式呼叫 

在本課程中，我們希望為我們的教育創業公司建立一個功能，讓用戶可以使用聊天機械人來尋找技術課程。我們將推薦符合他們技術水平、當前角色及感興趣技術的課程。 

為了完成這項工作，我們將結合使用： 
 - `OpenAI` 來為用戶創建聊天體驗
 - `Microsoft Learn Catalog API` 幫助用戶根據其需求查找課程 
 - `Function Calling` 將用戶的查詢傳送到一個函式以發出 API 請求。 

為了開始，讓我們先看看為何首先要使用函式呼叫： 


### 為甚麼要使用函數呼叫

如果你已經完成了本課程中的其他課程，你大概會明白使用大型語言模型（LLMs）的威力。希望你也能看到它們的一些限制。

函數呼叫是 OpenAI 服務的一個功能，旨在解決以下挑戰：

回應格式不一致：
- 在函數呼叫之前，來自大型語言模型的回應是無結構且不一致的。開發人員必須編寫複雜的驗證代碼來處理輸出中的每種變化。

與外部數據整合有限：
- 在此功能之前，將應用程式其他部分的數據整合到聊天上下文中是困難的。

透過標準化回應格式並實現與外部數據的無縫整合，函數呼叫簡化了開發流程，並減少了額外驗證邏輯的需求。

使用者無法獲得類似「斯德哥爾摩目前天氣如何？」這樣的即時答案。這是因為模型的資料僅限於訓練時的時間點。

讓我們看下面的範例來說明這個問題：

假設我們想建立一個學生資料庫，以便建議適合他們的課程。以下有兩個學生的描述，它們包含的資料非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此發送給大型語言模型 (LLM) 以解析資料。日後可以用於我們的應用程式，將資料發送到 API 或儲存到資料庫中。

讓我們建立兩個相同的提示，指示 LLM 我們感興趣的資訊：


我們想將此發送給大型語言模型（LLM）以解析對我們產品重要的部分。因此，我們可以創建兩個相同的提示詞來指導LLM：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


創建這兩個提示後，我們會使用 `client.responses.create` 將它們發送到 LLM。 我們把提示存入 `input` 變數，並將角色指定為 `user`。這是為了模仿用戶向聊天機械人寫入訊息。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-4o-mini"


: 

現在我們可以同時將兩個請求發送到 LLM，並檢查我們收到的回應。


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



儘管提示詞相同且描述類似，我們仍然可以得到不同格式的 `Grades` 屬性。 

如果你多次運行上述儲存格，格式可能會是 `3.7` 或 `3.7 GPA`。 

這是因為 LLM 以書寫的提示詞形式接收非結構化資料，並且返回的也是非結構化資料。我們需要有結構化格式，這樣才能知道在儲存或使用這些資料時該期待什麼。

透過使用函數呼叫，我們可以確保收到結構化的資料。使用函數呼叫時，LLM 實際上不會呼叫或執行任何函數。我們是為 LLM 創建一個回應要遵循的結構，然後使用這些結構化的回應來知道在應用程式中該執行哪個函數。  
 


![函數呼叫流程圖](../../../../translated_images/zh-HK/Function-Flow.083875364af4f4bb.webp)


然後我們可以將從函式返回的結果傳回給 LLM。LLM 會使用自然語言回應以回答用戶的查詢。


### 使用函數調用的案例

<strong>調用外部工具</strong>
聊天機械人非常擅長回答用戶的問題。通過使用函數調用，聊天機械人可以使用用戶的訊息來完成某些任務。例如，學生可以請聊天機械人「發送電郵給我的導師，說我需要更多這科的協助」。這可以調用一個函數 `send_email(to: string, body: string)`


**建立 API 或資料庫查詢**
用戶可以使用自然語言查找資訊，然後轉換成格式化的查詢或 API 請求。舉例來說，一位教師請求「誰完成了最後一份作業」就可以呼叫一個名稱為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數


<strong>建立結構化資料</strong>
用戶可以將一段文字或 CSV 使用大型語言模型 (LLM) 提取重要資訊。例如，學生可以將一篇有關和平協議的維基百科文章轉換成 AI 快閃卡。這可以透過一個函數 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 完成


## 2. 建立您的第一個函數調用

建立函數調用的過程包括3個主要步驟：
1. 使用您的函數列表和用戶訊息呼叫 Chat Completions API
2. 讀取模型的回應以執行動作，即執行函數或 API 呼叫
3. 使用您函數的回應再次呼叫 Chat Completions API，利用該資訊來建立對用戶的回應。


![函數調用流程](../../../../translated_images/zh-HK/LLM-Flow.3285ed8caf4796d7.webp)


### 函數呼叫的元素 

#### 用戶輸入 

第一步是建立用戶訊息。這可以通過取得文字輸入的值來動態指定，或者你可以在這裡分配一個值。如果這是你第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。 

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終用戶）。對於函數呼叫，我們將它指定為 `user` 並給出一個範例問題。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函數。

接下來我們會定義一個函數以及該函數的參數。我們將只使用一個名為 `search_courses` 的函數，但你可以建立多個函數。

<strong>重要</strong> ：函數會包含在系統訊息中發送給 LLM，並且會計入你可用的令牌數量中。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

函數定義結構有多個層次，每層都有其屬性。以下是巢狀結構的解析：

**頂層函數屬性：**

`name` - 我們想要呼叫的函數名稱。 

`description` - 函數的運作描述。在這裏，明確和清晰非常重要 

`parameters` - 模型回應中你希望輸出的值與格式列表 

**參數物件屬性：**

`type` - 參數物件的資料類型（通常為 "object"）

`properties` - 模型將用作回應的具體值列表 

**個別參數屬性：**

`name` - 由屬性鍵隱式定義（例如 "role"、"product"、"level"）

`type` - 該特定參數的資料類型（例如 "string"、"number"、"boolean"） 

`description` - 該特定參數的描述 

**選用屬性：**

`required` - 列出為完成函數呼叫而需要的參數陣列 


### 呼叫函數 
定義函數後，我們現在需要將它包含在 Chat Completion API 的呼叫中。我們透過在請求中加入 `functions` 來完成。在這裡是 `functions=functions`。 

也可以設定 `function_call` 為 `auto`。這表示我們會讓大型語言模型根據使用者訊息決定應該呼叫哪個函數，而不是自己指派。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們看看回應，並了解它是如何格式化的：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數名稱被呼叫，並且從用戶訊息中，LLM 能夠找到適合該函數參數的資料。


## 3.將函數調用整合到應用程式中。 


在測試了來自 LLM 的格式化回應後，現在我們可以將其整合到應用程式中。 

### 管理流程 

要將此整合到我們的應用程式中，讓我們採取以下步驟： 

首先，讓我們呼叫 OpenAI 服務並將訊息儲存在名為 `response_message` 的變數中。 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函數，用來呼叫 Microsoft Learn API 以取得課程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會檢查模型是否想要呼叫一個函式。之後，我們會從可用的函式中建立一個並配對到正在被呼叫的函式。 
接著，我們會將函式的參數映射到來自 LLM 的參數。

最後，我們會附加函式呼叫訊息以及 `search_courses` 訊息返回的值。這會提供 LLM 所需的全部資訊，
以便使用自然語言回應使用者。 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將把更新後的訊息發送給 LLM，這樣我們就能收到自然語言回答，而非 API JSON 格式的回應。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 編碼挑戰 

做得好！要繼續學習 OpenAI 函數呼叫，您可以構建：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多函數的參數可能會幫助學習者找到更多課程。您可以在這裡找到可用的 API 參數： 
 - 建立另一個函數呼叫，從學習者那裡獲取更多資訊，例如他們的母語 
 - 在函數呼叫和/或 API 呼叫沒有返回任何合適課程時建立錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
